In [13]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch.optim.lr_scheduler import ReduceLROnPlateau


config = {
    "dataset":{
        "dti":"../../Data/scope_onside_common_v3.parquet",
        "adr":"../../Data/final_rxnorm_meddra_v2.parquet"
    },
    "protein_emb_1":{
        "path":  "../../Data/3. Protein_enbeddings/ESM_embeddings_(t33_650m model).parquet",
        "id_col": "id", 
        "emb_col": "embedding"
    },
    "protein_emb_2":{
        "path": "../../Data/3. Protein_enbeddings/GVP-GNN_protein_embeddings.parquet",
        "id_col": "uniprot_id", 
        "emb_col": "embedding"
    },
    "drug_emb_1":{
        "path": "../../Data/2. Drug_embeddings/EGNN_drug_embeddings_v2.parquet", 
        "id_col": "drug_chembl_id", 
        "emb_col": "embedding"
    },
    "drug_emb_2":{
        "path": "../../Data/2. Drug_embeddings/smiles_embeddings_chemberta.parquet", 
        "id_col": "drug_chembl_id", 
        "emb_col": "embedding"
    }
}

In [14]:
dti_df = pd.read_parquet(config["dataset"]["dti"])
print(dti_df.info())

# copy selected columns to a new df
# drug_chembl_id as drug_id and target_uniprot_id as protein_id
dti_df = dti_df.rename(columns={"drug_chembl_id": "drug_id", "target_uniprot_id": "protein_id"})

df = dti_df.copy()
df = df[["drug_id", "protein_id", "label", "rxcui", "molfile_3d"]]


if config["protein_emb_1"]["path"]:
    protein_emb_1_df = pd.read_parquet(config["protein_emb_1"]["path"])
    protein_emb_1_df = protein_emb_1_df.rename(columns={config["protein_emb_1"]["id_col"]: "protein_id"})
    df = df.merge(protein_emb_1_df[["protein_id", config["protein_emb_1"]["emb_col"]]], on="protein_id", how="left")
    df = df.rename(columns={config["protein_emb_1"]["emb_col"]: "prot_emb_1"})

if config["protein_emb_2"]["path"]:
    protein_emb_2_df = pd.read_parquet(config["protein_emb_2"]["path"])
    protein_emb_2_df = protein_emb_2_df.rename(columns={config["protein_emb_2"]["id_col"]: "protein_id"})
    df = df.merge(protein_emb_2_df[["protein_id", config["protein_emb_2"]["emb_col"]]], on="protein_id", how="left")
    df = df.rename(columns={config["protein_emb_2"]["emb_col"]: "prot_emb_2"})

if config["drug_emb_1"]["path"]:
    drug_emb_1_df = pd.read_parquet(config["drug_emb_1"]["path"])
    drug_emb_1_df = drug_emb_1_df.rename(columns={config["drug_emb_1"]["id_col"]: "drug_id"})
    df = df.merge(drug_emb_1_df[["drug_id", config["drug_emb_1"]["emb_col"]]], on="drug_id", how="left")
    df = df.rename(columns={config["drug_emb_1"]["emb_col"]: "drug_emb_1"})

if config["drug_emb_2"]["path"]:
    drug_emb_2_df = pd.read_parquet(config["drug_emb_2"]["path"])
    drug_emb_2_df = drug_emb_2_df.rename(columns={config["drug_emb_2"]["id_col"]: "drug_id"})
    df = df.merge(drug_emb_2_df[["drug_id", config["drug_emb_2"]["emb_col"]]], on="drug_id", how="left")
    df = df.rename(columns={config["drug_emb_2"]["emb_col"]: "drug_emb_2"})



print(df.info())



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34741 entries, 0 to 34740
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   drug_chembl_id     34741 non-null  object
 1   target_uniprot_id  34741 non-null  object
 2   label              34741 non-null  int64 
 3   smiles             34741 non-null  object
 4   sequence           34741 non-null  object
 5   molfile_3d         34741 non-null  object
 6   rxcui              34741 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.9+ MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34741 entries, 0 to 34740
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   drug_id     34741 non-null  object
 1   protein_id  34741 non-null  object
 2   label       34741 non-null  int64 
 3   rxcui       34741 non-null  object
 4   molfile_3d  34741 non-null  object
 5   prot_emb_1  34741 non

In [15]:

class ADRData:
    def __init__(self, id_to_name_dict):
        """
        Initializes with a dictionary of {meddra_id: meddra_name}.
        """
        self.id_to_name = id_to_name_dict
        self.unique_ids = sorted(list(id_to_name_dict.keys()))
        
        self.id_to_idx = {adr_id: i for i, adr_id in enumerate(self.unique_ids)}
        self.idx_to_id = {i: adr_id for i, adr_id in enumerate(self.unique_ids)}
        
        self.vocab_size = len(self.unique_ids)

    def encode(self, adr_list):
        """
        Takes a list of ADR IDs and returns a binary vector (1s and 0s).
        Example: ['10028553', '10003041'] -> [0, 1, 0, 0, 1...]
        """
        vector = np.zeros(self.vocab_size, dtype=np.int8)
        
        for adr_id in adr_list:
            if adr_id in self.id_to_idx:
                idx = self.id_to_idx[adr_id]
                vector[idx] = 1
            else:
                print(f"Warning: ADR ID {adr_id} not in vocabulary.")
                
        return vector

    def decode(self, vector):
        """
        Takes a binary vector and returns a list of human-readable ADR names.
        """
        decoded_names = []
        
        active_indices = np.where(vector == 1)[0]
        
        for idx in active_indices:
            adr_id = self.idx_to_id[idx]
            name = self.id_to_name.get(adr_id, "Unknown ADR")
            decoded_names.append(name)
            
        return decoded_names

    def decode_indices(self, indices):
        """
        Takes a list or array of indices (e.g., [42, 105, 300]) 
        and returns the corresponding ADR names.
        """
        return [self.id_to_name.get(self.idx_to_id[idx], "Unknown ADR") for idx in indices]

    def decode_top_k(self, confidence_array, k=5):
        """
        Takes the raw probability array from the model, finds the top K 
        highest values, and returns names + their confidence scores.
        """
        # Get indices of the top k probabilities
        top_indices = np.argsort(confidence_array)[-k:][::-1]
        
        results = []
        for idx in top_indices:
            adr_id = self.idx_to_id[idx]
            name = self.id_to_name.get(adr_id, "Unknown ADR")
            conf = confidence_array[idx]
            results.append({"name": name, "confidence": round(float(conf), 4)})
            
        return results

    def decode_with_threshold(self, confidence_array, threshold=0.5):
        """
        Returns all ADRs that pass a specific confidence threshold.
        Useful for seeing everything the model is "sure" about.
        """
        active_indices = np.where(confidence_array >= threshold)[0]
        
        # Sort them by confidence (highest first)
        active_indices = active_indices[np.argsort(confidence_array[active_indices])[::-1]]
        
        return [self.id_to_name.get(self.idx_to_id[idx], "Unknown ADR") for idx in active_indices]


In [16]:
adrdf = pd.read_parquet(config['dataset']["adr"])
id_name_dict = dict(zip(adrdf['meddra_id'], adrdf['meddra_name']))
adr_manager = ADRData(id_name_dict)

In [17]:
drug_to_adr_list = adrdf.groupby('rxnorm_ingredient_id')['meddra_id'].apply(list).to_dict()

def get_encoded_adr(drug_id):
    # Get the list of ADRs for this drug, or an empty list if not found
    adrs = drug_to_adr_list.get(drug_id, [])
    return adr_manager.encode(adrs)

# 2. Map the drug_id (e.g., rxcui) to the encoded vector
# This will create a column where each cell is a numpy array
df['adr'] = df['rxcui'].map(get_encoded_adr)
print(f"Total rows with ADRs: {df['adr'].apply(lambda x: x.sum() > 0).sum()}")

Total rows with ADRs: 34741


In [18]:
import torch
from torch_geometric.data import Data
from rdkit import Chem
from tqdm import tqdm

def mol3d_to_graph(mol_block):
    """Converts a 3D Molblock string into a PyTorch Geometric Data object."""
    if not isinstance(mol_block, str) or mol_block.strip() == "":
        return None
        
    mol = Chem.MolFromMolBlock(mol_block, removeHs=True)
    if mol is None:
        return None
    
    # 1. Node Features (Atomic Number, Degree, Charge, Total Hs, IsAromatic)
    node_feats = []
    for atom in mol.GetAtoms():
        node_feats.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            atom.GetTotalNumHs(),
            int(atom.GetIsAromatic())
        ])
    x = torch.tensor(node_feats, dtype=torch.float)

    # 2. 3D Coordinates (pos)
    conf = mol.GetConformer()
    pos = torch.tensor(conf.GetPositions(), dtype=torch.float)

    # 3. Edge Index (Bonds)
    edge_indices = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_indices.append([i, j])
        edge_indices.append([j, i]) # Bi-directional for GNN
        
    # Handle cases with no bonds (rare for drugs, but safety first)
    if len(edge_indices) > 0:
        edge_index = torch.tensor(edge_indices, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)

    return Data(x=x, pos=pos, edge_index=edge_index)



In [19]:
# --- Execution Cell ---
print("Converting 3D Molfiles to Geometric Graphs...")
tqdm.pandas() # Enable progress bar for pandas
df['drug_graph'] = df['molfile_3d'].progress_apply(mol3d_to_graph)

# Check for failures
failed_count = df['drug_graph'].isna().sum()
if failed_count > 0:
    print(f"⚠️ Warning: {failed_count} molfiles could not be converted.")
else:
    print("✅ All drugs successfully converted to graphs.")

Converting 3D Molfiles to Geometric Graphs...


100%|██████████| 34741/34741 [00:23<00:00, 1476.57it/s]

✅ All drugs successfully converted to graphs.


In [23]:
# Select the first non-null graph
sample_graph = df[df['drug_graph'].notna()]['drug_graph'].iloc[0]

print("--- Sample Drug Graph Structure ---")
print(sample_graph)

print("\n--- Tensor Details ---")
print(f"Node Features (x) shape: {sample_graph.x.shape}")
print(f"3D Coordinates (pos) shape: {sample_graph.pos.shape}")
print(f"Edge Index shape: {sample_graph.edge_index.shape}")

# Show a few actual coordinate values to check scale
print("\n--- Sample Coordinates (First 10 atoms) ---")
print(sample_graph.pos[:10])

--- Sample Drug Graph Structure ---
Data(x=[27, 5], edge_index=[2, 58], pos=[27, 3])

--- Tensor Details ---
Node Features (x) shape: torch.Size([27, 5])
3D Coordinates (pos) shape: torch.Size([27, 3])
Edge Index shape: torch.Size([2, 58])

--- Sample Coordinates (First 10 atoms) ---
tensor([[ 4.7384,  2.4573,  1.5227],
        [ 5.5739,  1.5728,  1.6109],
        [ 6.3350,  1.4689,  2.7222],
        [ 5.8976,  0.4859,  0.5897],
        [ 5.1854,  0.5453, -0.6393],
        [ 3.8222,  0.1279, -0.5117],
        [ 3.1402,  0.2434, -1.8868],
        [ 1.7109, -0.0783, -1.9603],
        [ 0.8477,  0.8050, -1.1555],
        [ 0.2048,  0.1222,  0.0748]])


In [31]:
import os


PROTEIN_DIR = "../../AlphaFoldData/"

# 1. Get unique protein IDs from your dataframe
unique_protein_ids = df['protein_id'].unique()
print(f"Total unique proteins in DataFrame: {len(unique_protein_ids)}")

available_files = {f for f in os.listdir(PROTEIN_DIR) if f.endswith('.pdb')}
available_ids = {f.replace('.pdb', '') for f in available_files}

# 3. Separate IDs into available and missing
found_ids = [pid for pid in unique_protein_ids if pid in available_ids]
missing_ids = [pid for pid in unique_protein_ids if pid not in available_ids]

print(f"✅ Available AlphaFold files: {len(found_ids)}")
print(f"❌ Missing AlphaFold files: {len(missing_ids)}")

if missing_ids:
    print(f"Sample missing IDs: {missing_ids[:5]}")

df_final = df[df['protein_id'].isin(available_ids)].copy()
print(f"Final DataFrame size after filtering: {len(df_final)}")

Total unique proteins in DataFrame: 2385
✅ Available AlphaFold files: 2385
❌ Missing AlphaFold files: 0
Final DataFrame size after filtering: 34741


In [34]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA used by Torch: {torch.version.cuda}")
print(f"Is CUDA available?: {torch.cuda.is_available()}")

PyTorch version: 2.5.1
CUDA used by Torch: 12.4
Is CUDA available?: True


In [35]:
import torch
from torch_geometric.data import Data
from biopandas.pdb import PandasPdb
from torch_cluster import knn_graph
from tqdm import tqdm
import os

def process_protein_to_graph(protein_id, pdb_dir):
    """Extracts C-alpha backbone and creates a KNN graph for a protein."""
    pdb_path = os.path.join(pdb_dir, f"{protein_id}.pdb")
    
    try:
        # 1. Load PDB and filter for C-alpha (Backbone)
        ppdb = PandasPdb().read_pdb(pdb_path)
        df_atoms = ppdb.df['ATOM']
        ca_atoms = df_atoms[df_atoms['atom_name'] == 'CA'].copy()
        
        # 2. Map Residue Names to Indices (0-19 for standard AA, 20 for others)
        aa_map = {res: i for i, res in enumerate([
            'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
            'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
        ])}
        
        # 3. Create Tensors
        node_indices = torch.tensor([aa_map.get(res, 20) for res in ca_atoms['residue_name']], dtype=torch.long)
        pos = torch.tensor(ca_atoms[['x_coord', 'y_coord', 'z_coord']].values, dtype=torch.float)
        
        # 4. Create Graph Connectivity (K-Nearest Neighbors)
        # GVP and structural encoders need edges to pass messages
        edge_index = knn_graph(pos, k=10) 
        
        return Data(x=node_indices, pos=pos, edge_index=edge_index)
        
    except Exception as e:
        print(f"Error processing {protein_id}: {e}")
        return None



In [38]:
PROTEIN_DIR = "../../AlphaFoldData/"
CACHE_PATH = 'processed_protein_graphs.pt'
unique_proteins = df_final['protein_id'].unique()

# --- Cache Check Logic ---
if os.path.exists(CACHE_PATH):
    print(f"📦 Loading pre-processed protein graphs from {CACHE_PATH}...")
    protein_geometry_cache = torch.load(CACHE_PATH)
    
    # Check if all proteins in current DF are in the loaded cache
    missing_in_cache = [pid for pid in unique_proteins if pid not in protein_geometry_cache]
    
    if not missing_in_cache:
        print(f"✅ Cache is complete. Loaded {len(protein_geometry_cache)} proteins.")
    else:
        print(f"⚠️ Cache is missing {len(missing_in_cache)} proteins. Resuming preprocessing...")
        # Process only the missing ones
        for pid in tqdm(missing_in_cache):
            graph = process_protein_to_graph(pid, PROTEIN_DIR)
            if graph:
                protein_geometry_cache[pid] = graph
        
        # Update the saved file with the new complete cache
        torch.save(protein_geometry_cache, CACHE_PATH)
        print(f"✅ Cache updated and saved.")

else:
    print(f"🔍 No cache found. Starting fresh preprocessing for {len(unique_proteins)} proteins...")
    protein_geometry_cache = {}
    for pid in tqdm(unique_proteins):
        graph = process_protein_to_graph(pid, PROTEIN_DIR)
        if graph:
            protein_geometry_cache[pid] = graph
    
    torch.save(protein_geometry_cache, CACHE_PATH)
    print(f"✅ Preprocessing complete. Saved {len(protein_geometry_cache)} protein graphs.")

📦 Loading pre-processed protein graphs from processed_protein_graphs.pt...
✅ Cache is complete. Loaded 2385 proteins.


In [40]:
import torch
import torch.nn as nn
from torch_geometric.nn import global_mean_pool

class EGNNLayer(nn.Module):
    def __init__(self, n_feat):
        super().__init__()
        # Message function: phi_m(h_i, h_j, d_ij^2)
        self.edge_mlp = nn.Sequential(
            nn.Linear(n_feat * 2 + 1, n_feat),
            nn.SiLU(),
            nn.Linear(n_feat, n_feat)
        )
        # Node update function: phi_h(h_i, m_i)
        self.node_mlp = nn.Sequential(
            nn.Linear(n_feat * 2, n_feat),
            nn.SiLU(),
            nn.Linear(n_feat, n_feat)
        )

    def forward(self, h, pos, edge_index):
        row, col = edge_index
        # Compute squared distance d_ij^2
        dist = torch.sum((pos[row] - pos[col])**2, dim=-1, keepdim=True)
        
        # Message passing
        msg = self.edge_mlp(torch.cat([h[row], h[col], dist], dim=-1))
        
        # Aggregate
        aggr_msg = torch.zeros_like(h)
        aggr_msg.index_add_(0, row, msg)
        
        # Update node features
        h = h + self.node_mlp(torch.cat([h, aggr_msg], dim=-1))
        return h

class EGNNModule(nn.Module):
    def __init__(self, in_dim=5, hidden_dim=512, n_layers=3):
        super().__init__()
        self.embedding = nn.Linear(in_dim, hidden_dim)
        self.layers = nn.ModuleList([EGNNLayer(hidden_dim) for _ in range(n_layers)])
        
    def forward(self, data):
        # data: Batch of Drug Graphs (x, pos, edge_index, batch)
        h = self.embedding(data.x)
        for layer in self.layers:
            h = layer(h, data.pos, data.edge_index)
        return global_mean_pool(h, data.batch)

In [41]:
class GVPBlock(nn.Module):
    def __init__(self, in_dims, out_dims):
        super().__init__()
        self.si, self.vi = in_dims
        self.so, self.vo = out_dims
        
        self.wh = nn.Linear(self.vi, self.vo, bias=False)
        self.ws = nn.Linear(self.vi + self.si, self.so)
        
    def forward(self, x):
        s, v = x  # s: (N, si), v: (N, vi, 3)
        v_norm = torch.norm(v, dim=-1) # (N, vi)
        
        # Vector update
        v_out = self.wh(v.transpose(1, 2)).transpose(1, 2)
        
        # Scalar update (including vector norms as scalar info)
        s_out = self.ws(torch.cat([s, v_norm], dim=-1))
        
        # Gating: Scalars gate the vectors
        gating = torch.sigmoid(s_out[:, :self.vo]).unsqueeze(-1)
        v_out = v_out * gating
        
        return s_out, v_out

class GVPModule(nn.Module):
    def __init__(self, node_in_dim=21, hidden_dim=512):
        super().__init__()
        self.emb = nn.Embedding(node_in_dim, 64)
        # s_in=64, v_in=1 (pos) -> s_out=hidden, v_out=16
        self.gvp = GVPBlock((64, 1), (hidden_dim, 16))
        
    def forward(self, data):
        # data: Batch of Protein Graphs (x, pos, edge_index, batch)
        s = self.emb(data.x)
        v = data.pos.unsqueeze(1) # (N, 1, 3)
        
        s, v = self.gvp((s, v))
        return global_mean_pool(s, data.batch)

In [ ]:

class FusionModule(nn.Module):
    def __init__(self, dim1, dim2, output_dim):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(dim1 + dim2, output_dim),
            nn.LayerNorm(output_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
    def forward(self, e1, e2):
        return self.fusion(torch.cat([e1, e2], dim=1))

class MultiTaskFusionVAE(nn.Module):
    def __init__(self, drug_dims, prot_dims, adr_dim=4817, fused_dim=768, latent_dim=256):
        super().__init__()
        
        # 1. Structural Modules (3D)
        self.egnn_module = EGNNModule(in_dim=5, hidden_dim=drug_dims[0])
        self.gvp_module = GVPModule(node_in_dim=21, hidden_dim=prot_dims[0])

        # 2. Fusion Layers (Structural + Static)
        self.drug_fusion = FusionModule(drug_dims[0], drug_dims[1], fused_dim)
        self.prot_fusion = FusionModule(prot_dims[0], prot_dims[1], fused_dim)
        
        # 3. Rest of the Architecture
        self.context_encoder = nn.Sequential(
            nn.Linear(fused_dim * 2, 1536), # Expand slightly first
            nn.LayerNorm(1536),
            nn.ReLU(),
            nn.Dropout(0.1),                # Adding dropout because 768-D is prone to overfitting
            nn.Linear(1536, 1024),
            nn.ReLU(),
            nn.Linear(1024, 768)            # Feeds into fc_mu and fc_logvar
        )

        self.fc_mu = nn.Linear(768, latent_dim)
        self.fc_logvar = nn.Linear(768, latent_dim)
        
        self.adr_decoder = nn.Sequential(
            nn.Linear(latent_dim, 512), 
            nn.ReLU(), 
            nn.Linear(512, adr_dim)
        )
        self.dti_head = nn.Sequential(
            nn.Linear(latent_dim, 256), 
            nn.ReLU(), 
            nn.Linear(256, 1)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * torch.clamp(logvar, -10, 10))
        return mu + torch.randn_like(std) * std

    def forward(self, drug_graph, d2, prot_graph, p2):
        # Get 3D Structural Embeddings
        drug_3d = self.egnn_module(drug_graph)
        prot_3d = self.gvp_module(prot_graph)

        # Fuse with static embeddings (d2, p2)
        fused_drug = self.drug_fusion(drug_3d, d2) 
        fused_prot = self.prot_fusion(prot_3d, p2)
        
        # VAE Pipeline
        context = self.context_encoder(torch.cat([fused_drug, fused_prot], dim=1))
        mu, logvar = self.fc_mu(context), self.fc_logvar(context)
        z = self.reparameterize(mu, logvar)
        
        return self.dti_head(z), self.adr_decoder(z), mu, logvar

In [42]:
from sklearn.model_selection import train_test_split

# 1. Get all unique protein IDs
unique_proteins = df_final['protein_id'].unique()

# 2. Split protein IDs (not rows) to ensure no leakage
# We'll reserve 10% of proteins for Test and 10% for Validation
train_prot_ids, temp_prot_ids = train_test_split(
    unique_proteins, 
    test_size=0.20, 
    random_state=42
)

val_prot_ids, test_prot_ids = train_test_split(
    temp_prot_ids, 
    test_size=0.50, 
    random_state=42
)

# 3. Create the dataframes based on these ID splits
train_df = df_final[df_final['protein_id'].isin(train_prot_ids)]
val_df = df_final[df_final['protein_id'].isin(val_prot_ids)]
test_df = df_final[df_final['protein_id'].isin(test_prot_ids)]

# --- Verification & Metrics ---
print(f"--- Final Dataset Sizes ---")
print(f"Train Set: {len(train_df)} rows ({len(train_prot_ids)} proteins)")
print(f"Val Set:   {len(val_df)} rows ({len(val_prot_ids)} proteins) - [Model Selection]")
print(f"Test Set:  {len(test_df)} rows ({len(test_prot_ids)} proteins) - [Cold-Protein Eval]")

# 4. Recalculate Positive Weight for Training
num_neg = (train_df['label'] == 0).sum()
num_pos = (train_df['label'] == 1).sum()

# Avoid division by zero just in case
pos_weight_value = num_neg / num_pos if num_pos > 0 else 1.0

print(f"\nNew Positive Weight: {pos_weight_value:.2f}")
print(f"Positive/Negative Ratio in Train: 1:{num_neg/num_pos:.2f}")

--- Final Dataset Sizes ---
Train Set: 28055 rows (1908 proteins)
Val Set:   3335 rows (238 proteins) - [Model Selection]
Test Set:  3351 rows (239 proteins) - [Cold-Protein Eval]

New Positive Weight: 1.75
Positive/Negative Ratio in Train: 1:1.75


In [46]:
class MultiModalDTIDataset(Dataset):
    def __init__(self, df, protein_cache):
        self.df = df.reset_index(drop=True)
        self.protein_cache = protein_cache

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 1. Structural Data
        drug_graph = row['drug_graph']
        prot_graph = self.protein_cache[row['protein_id']]
        
        # 2. Static Embeddings (d2 and p2)
        d2 = torch.tensor(row['drug_emb_2'], dtype=torch.float)
        p2 = torch.tensor(row['prot_emb_2'], dtype=torch.float)
        
        # 3. Targets
        label = torch.tensor(row['label'], dtype=torch.float)
        
        # ADR targets (assuming they are stored as a list/array in 'adr_targets' column)
        adr_target = torch.tensor(row['adr'], dtype=torch.float)
        
        return drug_graph, d2, prot_graph, p2, label, adr_target

def collate_fn(batch):
    drug_graphs, d2_list, prot_graphs, p2_list, labels, adr_list = zip(*batch)
    
    # Geometric Batching
    drug_batch = Batch.from_data_list(drug_graphs)
    prot_batch = Batch.from_data_list(prot_graphs)
    
    # Tensor Stacking
    d2_batch = torch.stack(d2_list)
    p2_batch = torch.stack(p2_list)
    labels_batch = torch.stack(labels)
    adr_batch = torch.stack(adr_list)
    
    return drug_batch, d2_batch, prot_batch, p2_batch, labels_batch, adr_batch

In [47]:
from torch_geometric.loader import DataLoader

# --- Hyperparameters ---
BATCH_SIZE = 32 # Adjust based on your GPU RAM (16 or 32 is usually safe for GVP/EGNN)

# 1. Create Dataset Instances
train_dataset = MultiModalDTIDataset(train_df, protein_geometry_cache)
val_dataset = MultiModalDTIDataset(val_df, protein_geometry_cache)
test_dataset = MultiModalDTIDataset(test_df, protein_geometry_cache)

# 2. Create DataLoaders
# We use the custom collate_fn to merge the Drug/Protein graphs into Batches
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn,
    num_workers=0 # Set to 4+ if running on a local machine for speed
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn
)

print(f"✅ DataLoaders initiated.")
print(f"Number of Train Batches: {len(train_loader)}")
print(f"Number of Val Batches:   {len(val_loader)}")

✅ DataLoaders initiated.
Number of Train Batches: 877
Number of Val Batches:   105


In [48]:
# Pull one batch
db, d2, pb, p2, lbl, adr = next(iter(train_loader))

print("--- Batch Verification ---")
print(f"Drug Graph Batch: {db}")      # Should show 'batch' attribute
print(f"Protein Graph Batch: {pb}")   # Should show 'batch' attribute
print(f"Static Drug (d2): {d2.shape}") # Should be [BATCH_SIZE, drug_emb_2_dim]
print(f"Static Prot (p2): {p2.shape}") # Should be [BATCH_SIZE, prot_emb_2_dim]
print(f"Labels: {lbl.shape}")          # Should be [BATCH_SIZE]
print(f"ADR Targets: {adr.shape}")     # Should be [BATCH_SIZE, 4817]

--- Batch Verification ---
Drug Graph Batch: DataBatch(x=[1044, 5], edge_index=[2, 2256], pos=[1044, 3], batch=[1044], ptr=[33])
Protein Graph Batch: DataBatch(x=[24046], edge_index=[2, 240460], pos=[24046, 3], batch=[24046], ptr=[33])
Static Drug (d2): torch.Size([32, 384])
Static Prot (p2): torch.Size([32, 1024])
Labels: torch.Size([32])
ADR Targets: torch.Size([32, 4817])


In [54]:
# --- Your Custom Dimensions ---
DRUG_STRUC_OUT = 512   # EGNN output (increased to support larger latent)
DRUG_STATIC_IN = 384   # Your d2 size

PROT_STRUC_OUT = 1024  # GVP output (increased to match p2)
PROT_STATIC_IN = 1024  # Your p2 size

FUSED_DIM = 1024       # Your request
LATENT_DIM = 768       # Your request
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize the model with these high-capacity dims
model = MultiTaskFusionVAE(
    drug_dims=[DRUG_STRUC_OUT, DRUG_STATIC_IN],
    prot_dims=[PROT_STRUC_OUT, PROT_STATIC_IN],
    adr_dim=4817,
    fused_dim=FUSED_DIM,
    latent_dim=LATENT_DIM
).to(device)

print(f"🚀 Model initialized with {LATENT_DIM}-D Latent Space.")

🚀 Model initialized with 768-D Latent Space.


In [55]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import torch
import numpy as np

def evaluate_multi_task(model, loader, device, dti_threshold=0.5, adr_threshold=0.5):
    model.eval()
    
    # Storage for predictions and ground truth
    all_dti_probs = []
    all_dti_labels = []
    all_adr_probs = []
    all_adr_labels = []

    print("Running Evaluation...")
    with torch.no_grad():
        for db, d2, pb, p2, lbl, adr in loader:
            db, d2, pb, p2 = db.to(device), d2.to(device), pb.to(device), p2.to(device)
            
            # Forward pass
            dti_logits, adr_logits, _, _ = model(db, d2, pb, p2)
            
            # Convert to probabilities
            dti_probs = torch.sigmoid(dti_logits).cpu().numpy()
            adr_probs = torch.sigmoid(adr_logits).cpu().numpy()
            
            all_dti_probs.extend(dti_probs)
            all_dti_labels.extend(lbl.cpu().numpy())
            all_adr_probs.extend(adr_probs)
            all_adr_labels.extend(adr.cpu().numpy())

    # Convert to arrays
    all_dti_probs = np.array(all_dti_probs).flatten()
    all_dti_labels = np.array(all_dti_labels).flatten()
    all_adr_probs = np.vstack(all_adr_probs)
    all_adr_labels = np.vstack(all_adr_labels)

    # --- DTI Metrics ---
    dti_auroc = roc_auc_score(all_dti_labels, all_dti_probs)
    dti_auprc = average_precision_score(all_dti_labels, all_dti_probs)
    dti_preds = (all_dti_probs > dti_threshold).astype(int)
    dti_f1 = f1_score(all_dti_labels, dti_preds)

    # --- ADR Metrics (Macro-averaged over 4817 tasks) ---
    # We use 'macro' to ensure rare side effects are weighted equally to common ones
    adr_auroc = roc_auc_score(all_adr_labels, all_adr_probs, average='macro')
    adr_auprc = average_precision_score(all_adr_labels, all_adr_probs, average='macro')
    adr_preds = (all_adr_probs > adr_threshold).astype(int)
    adr_f1 = f1_score(all_adr_labels, adr_preds, average='macro', zero_division=0)

    return {
        'DTI': {'AUROC': dti_auroc, 'AUPRC': dti_auprc, 'F1': dti_f1},
        'ADR': {'AUROC': adr_auroc, 'AUPRC': adr_auprc, 'F1': adr_f1}
    }

In [61]:
import torch
import torch.nn.functional as F
from tqdm import tqdm

def train_structural_vae(model, train_loader, val_loader, optimizer, scheduler, device, pos_weight, epochs=100, patience=10):
    best_dti_auprc = 0.0  # We track AUPRC for Cold-Protein selection
    early_stop_counter = 0
    
    w_dti, w_adr, w_kl = 1.0, 20, 0.005 

    for epoch in range(epochs):
        # --- 1. TRAINING PHASE ---
        model.train()
        train_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        
        for db, d2, pb, p2, lbl, adr in pbar:
            db, d2, pb, p2 = db.to(device), d2.to(device), pb.to(device), p2.to(device)
            lbl, adr = lbl.to(device), adr.to(device)
            
            optimizer.zero_grad()
            dti_logits, adr_logits, mu, logvar = model(db, d2, pb, p2)
            
            # Loss Components
            loss_dti = F.binary_cross_entropy_with_logits(dti_logits.squeeze(), lbl, pos_weight=torch.tensor([pos_weight]).to(device))
            loss_adr = F.binary_cross_entropy_with_logits(adr_logits, adr)
            loss_kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / mu.size(0)
            
            batch_loss = (w_dti * loss_dti) + (w_adr * loss_adr) + (w_kl * loss_kl)
            batch_loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += batch_loss.item()
            pbar.set_postfix({'loss': f"{batch_loss.item():.4f}"})

        # --- 2. EVALUATION PHASE (Calling your Evaluate Function) ---
        # We run evaluation on the Validation Set (Cold-Protein)
        metrics = evaluate_multi_task(model, val_loader, device)
        
        dti_results = metrics['DTI']
        adr_results = metrics['ADR']
        
        print(f"\n--- Epoch {epoch+1} Metrics ---")
        print(f"DTI -> AUROC: {dti_results['AUROC']:.4f} | AUPRC: {dti_results['AUPRC']:.4f} | F1: {dti_results['F1']:.4f}")
        print(f"ADR -> AUROC: {adr_results['AUROC']:.4f} | AUPRC: {adr_results['AUPRC']:.4f} | F1: {adr_results['F1']:.4f}")

        # --- 3. MODEL SELECTION & EARLY STOPPING ---
        # In DTI research, AUPRC is usually the best metric for model selection
        current_val_score = 0.6*dti_results['AUPRC'] + 0.4 * adr_results['AUPRC']
        
        scheduler.step(1 - current_val_score) # Step scheduler based on AUPRC improvement

        if current_val_score > best_dti_auprc:
            best_dti_auprc = current_val_score
            torch.save(model.state_dict(), 'best_structural_vae_model.pt')
            early_stop_counter = 0
            print(f"⭐ New Best AUPRC: {best_dti_auprc:.4f}! Model saved.")
        else:
            early_stop_counter += 1
            print(f"No improvement for {early_stop_counter} epochs.")
            if early_stop_counter >= patience:
                print("🛑 Early stopping triggered.")
                break
                
    return model

In [62]:
# AdamW is preferred for high-capacity structural models
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)

# Scheduler helps "fine-tune" as the model approaches convergence
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=3, verbose=True
)

In [63]:
history = train_structural_vae(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    pos_weight=pos_weight_value,
    epochs=100,
    patience=10
)

Epoch 1/100 [Train]: 100%|██████████| 877/877 [00:25<00:00, 34.34it/s, loss=2.1152]


Running Evaluation...

--- Epoch 1 Metrics ---
DTI -> AUROC: 0.7178 | AUPRC: 0.5315 | F1: 0.5369
ADR -> AUROC: nan | AUPRC: 0.0243 | F1: 0.0049
⭐ New Best AUPRC: 0.3286! Model saved.


Epoch 2/100 [Train]: 100%|██████████| 877/877 [00:25<00:00, 34.86it/s, loss=2.3363]


Running Evaluation...

--- Epoch 2 Metrics ---
DTI -> AUROC: 0.6962 | AUPRC: 0.4659 | F1: 0.5504
ADR -> AUROC: nan | AUPRC: 0.0240 | F1: 0.0056
No improvement for 1 epochs.


Epoch 3/100 [Train]: 100%|██████████| 877/877 [00:25<00:00, 34.70it/s, loss=2.0164]


...

Running Evaluation...

--- Epoch 7 Metrics ---
DTI -> AUROC: 0.7293 | AUPRC: 0.4830 | F1: 0.5915
ADR -> AUROC: nan | AUPRC: 0.0254 | F1: 0.0051
No improvement for 6 epochs.


Epoch 8/100 [Train]: 100%|██████████| 877/877 [00:26<00:00, 32.84it/s, loss=1.5484]


Running Evaluation...

--- Epoch 8 Metrics ---
DTI -> AUROC: 0.7304 | AUPRC: 0.4730 | F1: 0.5800
ADR -> AUROC: nan | AUPRC: 0.0249 | F1: 0.0057
No improvement for 7 epochs.


Epoch 9/100 [Train]:  69%|██████▊   | 602/877 [00:18<00:08, 32.88it/s, loss=1.8930]


KeyboardInterrupt: 